# Order Analysis — пакетная обработка CSV-файлов с заказами

**Тип проекта:** автоматизация обработки данных на Python  
**Основные инструменты:** Python, pandas, dataclasses, logging, pathlib

## Цель проекта

Разработать программу, которая обрабатывает все CSV-файлы из папки `data`, оставляет заказы со статусом `Delivered`, рассчитывает основные показатели по каждому файлу и сохраняет общий отчёт в CSV.

Программа также должна пропускать файлы, которые не удалось обработать, записывать информацию об ошибках в лог и продолжать обработку остальных файлов.

> Этот Notebook объясняет устройство и логику готового проекта. Основной сценарий запускается из `run.py`; Notebook не заменяет исходную программу.

## 1. Структура проекта

```text
order_analysis/
├── data/                   # входные CSV-файлы
├── reports/                # итоговый отчёт
├── logs/                   # журнал ошибок
├── src/
│   └── analyzer.py         # класс OrderAnalyzer и логика обработки
├── config.py               # пути и параметры
├── run.py                  # точка входа
├── requirements.txt        # зависимости
└── order_analysis.ipynb    # описание проекта
```

Разделение по файлам помогает держать настройки, обработку данных и запуск программы отдельно друг от друга.

## 2. Настройки

В `config.py` вынесены пути к папкам, имена выходных файлов, значение статуса доставленного заказа и параметры чтения CSV. Благодаря этому часть настроек можно менять без редактирования логики класса.

Фрагмент `config.py`:

```python
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
REPORTS_DIR = BASE_DIR / "reports"
LOGS_DIR = BASE_DIR / "logs"

SUMMARY_REPORT_FILENAME = "summary_report.csv"
ERROR_LOG_FILENAME = "errors.log"

STATUS_COLUMN = "status"
DELIVERED_STATUS_VALUE = "Delivered"

CSV_READ_KWARGS = {
    "sep": ",",
    "dtype": "object",
    "keep_default_na": True
}
```

Это фрагмент исходного конфигурационного файла. Он приведён как пример и не требует запуска в Notebook: переменная `__file__` задаётся при выполнении Python-файла, а не в обычной ячейке Notebook.

Здесь приведён фрагмент `config.py` для знакомства с настройками. В самом Notebook переменная `__file__` не определена, поэтому этот фрагмент предназначен для чтения, а не для самостоятельного запуска в ячейке.

## 3. Результат обработки одного файла

Для хранения результатов используется `dataclass` `FileResult`. Он группирует четыре значения: имя обработанного файла, суммарную выручку, средний чек и количество доставленных заказов.

Такой объект удобнее, чем передавать несколько отдельных значений: у каждого поля есть понятное имя.

Ключевые методы из `src/analyzer.py` выглядят так:

```python
def filter_delivered(self, df):
    return df[df[self.status_column] == self.delivered_status_value].copy()

def calculate_metrics(self, delivered_df):
    orders_count = len(delivered_df)

    total_amount_numeric = pd.to_numeric(
        delivered_df["total_amount"],
        errors="coerce"
    )
    total_revenue = float(total_amount_numeric.sum(skipna=True))
    avg_check = (
        float(total_amount_numeric.mean(skipna=True))
        if orders_count > 0 else 0.0
    )

    if pd.isna(avg_check):
        avg_check = 0.0

    return total_revenue, avg_check, orders_count
```

Это методы класса: `self` ссылается на текущий объект `OrderAnalyzer`, поэтому фрагмент рассматривается как документация к реализации, а не как самостоятельная ячейка.

## 4. Алгоритм обработки

Класс `OrderAnalyzer` разделяет задачу на небольшие методы с разными обязанностями:

1. `load_csv()` читает файл и проверяет наличие обязательных колонок.
2. `filter_delivered()` оставляет строки со статусом `Delivered`.
3. `calculate_metrics()` рассчитывает показатели.
4. `process_one_file()` объединяет эти шаги для одного файла.
5. `process_all_files()` перебирает CSV-файлы, собирает результаты и сохраняет общий отчёт.

Такое разделение облегчает чтение и сопровождение кода.

In [ ]:
# Основная логика из src/analyzer.py — фрагмент для иллюстрации

def filter_delivered(self, df):
    return df[df[self.status_column] == self.delivered_status_value].copy()

def calculate_metrics(self, delivered_df):
    orders_count = len(delivered_df)

    total_amount_numeric = pd.to_numeric(
        delivered_df["total_amount"],
        errors="coerce"
    )
    total_revenue = float(total_amount_numeric.sum(skipna=True))
    avg_check = (
        float(total_amount_numeric.mean(skipna=True))
        if orders_count > 0 else 0.0
    )

    if pd.isna(avg_check):
        avg_check = 0.0

    return total_revenue, avg_check, orders_count

В `calculate_metrics()` значение `total_amount` приводится к числовому типу. `errors="coerce"` превращает значения, которые нельзя преобразовать в число, в пропуски; при расчёте суммы и среднего они пропускаются.

При интерпретации результата важно помнить: `orders_count` считается по количеству строк после фильтрации, а сумма и среднее — по числовым значениям `total_amount`. Поэтому качество исходных данных влияет на интерпретацию метрик.

## 5. Обработка ошибок и пакетный запуск

В `process_all_files()` каждый файл обрабатывается внутри `try/except`. Если чтение или проверка конкретного файла завершается ошибкой, программа увеличивает счётчик ошибок, записывает сообщение в лог и переходит к следующему файлу.

После обхода папки результаты успешно обработанных файлов собираются в таблицу и сохраняются в `reports/summary_report.csv`.

Для запуска из терминала перейдите в папку `order_analysis` и выполните:

```bash
python run.py
```

Зависимость проекта указана в `requirements.txt`. Установка:

```bash
pip install -r requirements.txt
```

## 6. Просмотр сформированного отчёта

Следующая ячейка читает уже сохранённый отчёт из папки `reports`. Она не запускает повторную обработку исходных файлов и не изменяет проект.

In [ ]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "reports" / "summary_report.csv").exists():
    # Если Notebook открыт из корневой папки проекта на уровень выше
    project_dir = project_dir / "order_analysis"

report_path = project_dir / "reports" / "summary_report.csv"
report = pd.read_csv(report_path)
report

### Как читать показатели

- `file_name` — имя входного файла.
- `total_revenue` — сумма `total_amount` по заказам со статусом `Delivered`.
- `avg_check` — среднее значение `total_amount` для отобранных строк с учётом числового преобразования.
- `orders_count` — количество строк со статусом `Delivered`.

Отчёт содержит отдельную строку на каждый файл, который программа успешно обработала.

## 7. Проверка обработки ошибок

В проекте предусмотрены обработка исключений при чтении/проверке файла и запись сообщений в `logs/errors.log`. В архиве также находится файл `order_broken.csv`, предназначенный для проверки обработки некорректных данных.

При этом сохранённый в архиве `summary_report.csv` содержит строку для `order_broken.csv`, а `errors.log` пуст. Это означает, что в последнем сохранённом результате этот файл был обработан как успешный и не вызвал исключения. Одного названия `order_broken.csv` недостаточно, чтобы считать тест неудачным или успешным: важно, какую именно ошибку должен выявлять тест и действительно ли она присутствует в содержимом файла.

Для демонстрации отказоустойчивости можно отдельно проверить как минимум два случая:
- пустой CSV-файл;
- CSV-файл без обязательной колонки `status` или `total_amount`.

После запуска стоит проверить, что некорректный файл не попал в отчёт, сообщение записано в `logs/errors.log`, а остальные корректные файлы обработаны.

In [ ]:
# Просмотр текущего отчёта и журнала ошибок

from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "reports" / "summary_report.csv").exists():
    project_dir = project_dir / "order_analysis"

report_path = project_dir / "reports" / "summary_report.csv"
log_path = project_dir / "logs" / "errors.log"

print("Итоговый отчёт:")
display(pd.read_csv(report_path))

print("\nСодержимое журнала ошибок:")
if log_path.exists():
    log_text = log_path.read_text(encoding="utf-8")
    print(log_text if log_text.strip() else "Журнал ошибок пуст.")
else:
    print("Файл журнала пока не создан.")

## 8. Итоги

В проекте реализована пакетная обработка CSV-файлов с помощью Python и pandas. Логика организована в классе `OrderAnalyzer`, параметры вынесены в конфигурационный файл, результаты одного файла представлены через `dataclass`, а ошибки отдельных файлов обрабатываются отдельно от общего цикла.

### Какие навыки демонстрирует проект

- чтение и обработка CSV с pandas;
- фильтрация данных по условию;
- расчёт агрегированных показателей;
- организация кода с классами и методами;
- использование `dataclass` для структурирования результатов;
- настройка путей через `pathlib`;
- логирование и обработка исключений;
- формирование итогового CSV-отчёта.

### Возможное улучшение

В качестве небольшого улучшения можно вынести имя колонки `total_amount` в конфигурацию. Это сделает настройку колонок более последовательной и уменьшит зависимость класса от жёстко заданных имён.

**Вывод:** проект показывает не только работу с таблицами, но и базовые навыки построения небольшого автоматизированного инструмента для пакетной обработки данных.